# Intégration de LangChain avec LlamaIndex

## Summary

This notebook demonstrates how to build an **intelligent multi-tool AI agent** by combining two powerful frameworks:

- **LlamaIndex** is used to load PDF documents, split them into chunks, generate vector embeddings via OpenAI, and store them in a semantic search index.
- **LangChain + LangGraph** is used to create a **ReAct agent** that can reason over multiple tools to answer any question.

The agent has access to two tools:
1. **PDF Search** (`ai_pdfs`) — queries the LlamaIndex vector store to retrieve answers from the indexed documents.
2. **Web Search** (`DuckDuckGoSearch`) — performs real-time web searches for questions not covered by the PDFs.

At each step, the agent decides which tool to use based on the question, calls it, observes the result, and repeats until it has a final answer.

**Tech stack:** LlamaIndex · LangChain · LangGraph · OpenAI GPT-4o-mini · DuckDuckGo

---

## Architecture globale
1. **LlamaIndex** charge des PDFs, génère des embeddings et crée un index vectoriel
2. L'index est exposé comme un **outil LangChain**
3. Un **agent ReAct** (LangGraph) combine cet outil avec une recherche web (DuckDuckGo) pour répondre à n'importe quelle question

## Prérequis
- Une clé API OpenAI valide
- Des fichiers PDF à indexer (adapter les chemins dans la cellule 3)

---

## 1. Installation des dépendances

In [12]:
!pip install llama-index
!pip install duckduckgo-search langchain-community
!pip install -U ddgs
!pip install langchain-openai
!pip install langgraph
!pip install --upgrade langchain langchain-core langchain-community langchain-openai

## 2. Configuration de la clé API OpenAI

On définit la clé API OpenAI comme variable d'environnement.  
`getpass` est utilisé pour saisir la clé de façon **sécurisée** (non visible à l'écran).

In [2]:
import warnings
import os
from kaggle_secrets import UserSecretsClient
warnings.filterwarnings('ignore')

user_secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret("OPENAI_API_KEY")

## 3. Chargement des PDFs et création de l'index vectoriel

`SimpleDirectoryReader` lit les PDFs, les découpe en chunks, puis `VectorStoreIndex` génère des **embeddings** via OpenAI et les stocke pour la recherche sémantique.

> ⚠️ **Adaptez les chemins** dans `pdf_files` à votre environnement local.  
> `%%capture` masque les logs de chargement pour garder la sortie propre.

In [3]:
%%capture
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
pdf_files = ["/kaggle/input/cours-acp/cours-acp.pdf", "/kaggle/input/cours-acp/cours-afc.pdf"]
docs = SimpleDirectoryReader(input_files= pdf_files).load_data()
index = VectorStoreIndex.from_documents(docs)

## 4. Test de l'index — Requête directe

On vérifie que l'index fonctionne en lui posant directement une question sur l'ACP, **sans passer par l'agent**.

In [4]:
query_engine = index.as_query_engine()
response = query_engine.query("Qu'est-ce que l'Analyse en Composantes Principales ?")
print(response)

L'Analyse en Composantes Principales est une méthode statistique utilisée pour réduire la dimensionnalité des données en transformant un ensemble de variables corrélées en un nouvel ensemble de variables non corrélées appelées composantes principales. Ces composantes principales capturent l'essentiel de l'information contenue dans les données d'origine, permettant ainsi de simplifier l'analyse et de visualiser les relations entre les individus et les variables de manière plus efficace.


## 5. Sauvegarde de l'index sur disque

Pour éviter de recréer l'index à chaque exécution (opération coûteuse en temps et en tokens), on le **persiste** dans un dossier local.

In [6]:
# Sauvegarder l'index dans le dossier "mon_index_sauvegarde"
index.storage_context.persist(persist_dir="mon_index_sauvegarde")

## 6. Rechargement de l'index depuis le disque

On recharge l'index précédemment sauvegardé, **sans retraiter les PDFs**. Utile pour les exécutions suivantes.

In [7]:
from llama_index.core import StorageContext, load_index_from_storage

PERSIST_DIR = "mon_index_sauvegarde"

if not os.path.exists(PERSIST_DIR):
    raise FileNotFoundError(
        f"Le dossier '{PERSIST_DIR}' est introuvable. "
    )

# Reconstruire le contexte de stockage depuis le dossier
storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)

# Charger l'index en mémoire
index = load_index_from_storage(storage_context)

print(f"✅ Index rechargé avec succès depuis '{PERSIST_DIR}'")

✅ Index rechargé avec succès depuis 'mon_index_sauvegarde'


## 7. Conversion de l'index en outil LangChain

On encapsule le query engine LlamaIndex dans un `QueryEngineTool`, puis on le convertit en outil LangChain via `.to_langchain_tool()`.  
L'agent pourra ainsi **interroger les PDFs comme un outil natif LangChain**.

In [8]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

ai_engine = index.as_query_engine(similarity_top_k=3)

query_tool = QueryEngineTool(
    query_engine=ai_engine,
    metadata=ToolMetadata(
        name="ai_pdfs",
        description="Answers questions based on the uploaded AI PDF documents."
    ),
)

tool = query_tool.to_langchain_tool()

## 8. Ajout de la recherche web DuckDuckGo

On ajoute un second outil : la recherche web via **DuckDuckGo** (gratuit, sans clé API).  
L'agent dispose maintenant de 2 outils :
- `ai_pdfs` → recherche dans les PDFs indexés
- `DuckDuckGoSearch` → recherche en temps réel sur le web

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
tools = [tool, search_tool]

## 9. Création de l'agent LangChain (ReAct)

On crée un agent **ReAct** (Reasoning + Acting) avec GPT-4o-mini.  
L'agent raisonne étape par étape pour décider quel outil utiliser :
1. Il reçoit une question
2. Il **réfléchit** (Thought) au meilleur outil à utiliser
3. Il **agit** (Action) en appelant l'outil
4. Il **observe** (Observation) la réponse
5. Il répète jusqu'à avoir une réponse finale

In [13]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Initialiser le LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Créer l'agent ReAct avec LangGraph (approche recommandée LangChain 0.3+)
# LangGraph gère automatiquement la boucle Thought → Action → Observation
agent_executor = create_react_agent(model=llm, tools=tools)

print("✅ Agent créé avec succès")

✅ Agent créé avec succès


## 10. Interrogation de l'agent

On pose une question à l'agent. Il va automatiquement **décider quel outil utiliser** :
- Si la réponse est dans les PDFs → il utilise `ai_pdfs`
- Si c'est une question générale → il utilise `DuckDuckGoSearch`

In [14]:
result = agent_executor.invoke({"messages": [("human", "Explique-moi la théorie de la relativité.")]})
print(result["messages"][-1].content)

La théorie de la relativité, développée par Albert Einstein au début du 20ème siècle, se divise en deux parties principales : la relativité restreinte et la relativité générale.

### 1. Relativité Restreinte (1905)
La relativité restreinte repose sur deux postulats fondamentaux :
- **Principe de relativité** : Les lois de la physique sont les mêmes pour tous les observateurs, peu importe leur état de mouvement, tant qu'ils se déplacent à une vitesse constante (c'est-à-dire qu'ils sont en mouvement uniforme).
- **Invariance de la vitesse de la lumière** : La vitesse de la lumière dans le vide est constante et indépendante de la vitesse de la source lumineuse ou de l'observateur. Elle est d'environ 299 792 km/s.

#### Conséquences de la relativité restreinte :
- **Dilataion du temps** : Le temps passe plus lentement pour un objet en mouvement par rapport à un observateur au repos. Cela signifie que si un jumeau voyage dans l'espace à une vitesse proche de celle de la lumière, il vieillir